In [1]:
# Базовые библиотеки для воспроизводимости, работы с данными и удобного вывода результатов.
import os
import re
import sys
import random
import subprocess
import sentence_transformers
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def safe_ensure_package(package_name: str, import_name: Optional[str] = None) -> bool:
    """Пытается импортировать пакет и при необходимости установить его через pip.
    Если установка не удалась, возвращает False, но не роняет ноутбук.
    """
    target = import_name or package_name
    try:
        __import__(target)
        return True
    except Exception:
        print(f"Пробуем установить пакет: {package_name}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
            __import__(target)
            return True
        except Exception as e:
            print(f"Не удалось подготовить пакет {package_name}: {e!r}")
            return False


# Для retrieval-контура попробуем установить основные зависимости.
# Даже если sentence-transformers не поднимется, ноутбук сможет работать через fallback.
FAISS_READY = safe_ensure_package("faiss-cpu", "faiss")
SENTENCE_TRANSFORMERS_READY = safe_ensure_package("sentence-transformers", "sentence_transformers")


try:
    import faiss  # type: ignore
    FAISS_AVAILABLE = True
except Exception as e:
    FAISS_AVAILABLE = False
    print("FAISS недоступен, будет использован fallback на sklearn NearestNeighbors.")
    print("Причина:", repr(e))



print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS available:", FAISS_AVAILABLE)
print("sentence-transformers доступен:", SENTENCE_TRANSFORMERS_READY)
print(f"Python: {sys.version.split()[0]}")
print(f"- faiss / faiss-cpu: {faiss.__version__}")
print(f"- sentence-transformers: {sentence_transformers.__version__}")

# Фиксируем seed и определяем устройство.
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except:
        pass

set_seed(42)

try:
    import torch

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Устройство для работы:", DEVICE)

NumPy: 2.0.2
Pandas: 2.2.2
FAISS available: True
sentence-transformers доступен: True
Python: 3.12.3
- faiss / faiss-cpu: 1.13.2
- sentence-transformers: 5.4.0
Устройство для работы: cpu


In [2]:
kb_path = "data/documents.csv"
docs_df = pd.read_csv(kb_path)

# Приводим к формату словарей, ожидаемому конвейером чанкинга
documents = docs_df.to_dict(orient="records")

# 2. Sanity-check и вывод примеров
print(f"📦 Загружено документов: {len(documents)}")
display(Markdown("### Первые 5 документов базы знаний"))
display(docs_df.head(5))

📦 Загружено документов: 35


### Первые 5 документов базы знаний

,doc_id,title,text
0,req_01,Установка и первые шаги,Библиотеку requests можно установить командой ...
1,req_02,Базовый GET-запрос,Метод requests.get(url) отправляет HTTP-запрос...
2,req_03,Передача параметров в URL,"Параметр params принимает словарь, который авт..."
3,req_04,Отправка POST-запросов,Для отправки данных на сервер используется req...
4,req_05,Работа с JSON,Передача JSON выполняется через параметр json=...


Предметная область
База знаний содержит справочные материалы по популярной Python-библиотеке `requests`. 
Тема выбрана потому, что:
- API библиотеки хорошо структурирован и содержит чёткие сценарии (GET, POST, сессии, таймауты, ошибки);
- Тексты достаточно информативны для осмысленного семантического поиска;
- Подходит для демонстрации учебного RAG-контура без необходимости поднимать внешние API или парсить сложный веб-контент.

In [3]:
def chunk_text(text: str, chunk_size: int = 25, overlap: int = 5) -> List[str]:
    """Разбиение текста на фрагменты по словам с перекрытием."""
    words = text.split()
    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")
        
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]
        if not chunk_words:
            continue
        chunks.append(" ".join(chunk_words))
        if end >= len(words):
            break
    return chunks

# Применяем чанкинг
def build_chunks(docs: List[Dict[str, str]], chunk_size: int, overlap: int) -> pd.DataFrame:
    rows = []
    for doc in docs:
        parts = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for idx, chunk_txt in enumerate(parts):
            rows.append({
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "chunk_id": f"{doc['doc_id']}_c{idx}",
                "chunk_text": chunk_txt
            })
    return pd.DataFrame(rows)

# Показываем пример чанкинга
example_chunks = build_chunks([docs_df.iloc[3].to_dict()], chunk_size=12, overlap=4)
display(Markdown("### Пример чанкинга (документ про JSON)"))
display(example_chunks)

### Пример чанкинга (документ про JSON)

,doc_id,title,chunk_id,chunk_text
0,req_04,Отправка POST-запросов,req_04_c0,Для отправки данных на сервер используется req...
1,req_04,Отправка POST-запросов,req_04_c1,Параметр data кодируется как application/x-www...


**Параметры:** `chunk_size=12`, `overlap=4`. Выбраны эмпирически: размер достаточен для сохранения смысла предложения, а перекрытие в 30% предотвращает потерю контекста на границах фрагментов.

In [4]:
class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray: raise NotImplementedError
    def encode_queries(self, texts: List[str]) -> np.ndarray: raise NotImplementedError

class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu"):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"ST: {model_name}"
        
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(texts, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    def encode_queries(self, texts: List[str]) -> np.ndarray:
        return self.fit_documents(texts)

class TfidfBackend(EmbeddingBackend):
    def __init__(self):
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.backend_name = "TF-IDF Fallback"
        
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.fit_transform(texts).toarray().astype(np.float32)
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms
    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.transform(texts).toarray().astype(np.float32)
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

def select_backend(device: str = "cpu"):
    try:
        return SentenceTransformersBackend("sentence-transformers/all-MiniLM-L6-v2", device=device)
    except Exception as e:
        print(f"Dense-модель недоступна ({e}). Переключаемся на TF-IDF.")
        return TfidfBackend()

def build_faiss_index(backend, chunks_df: pd.DataFrame):
    vectors = backend.fit_documents(chunks_df["chunk_text"].tolist())
    if not FAISS_AVAILABLE:
        raise RuntimeError("FAISS не установлен.")
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index, vectors

# Сборка конвейера
backend = select_backend(DEVICE)
chunks_df = build_chunks(documents, chunk_size=25, overlap=6)
faiss_index, chunk_vectors = build_faiss_index(backend, chunks_df)

print(f"Backend: {backend.backend_name}")
print(f"Всего чанков в индексе: {len(chunks_df)}")

def search_chunks(query: str, index, backend, chunks_df, top_k: int = 3) -> pd.DataFrame:
    q_vec = backend.encode_queries([query]).astype(np.float32)
    scores, indices = index.search(q_vec, top_k)
    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        row = chunks_df.iloc[int(idx)]
        rows.append({"rank": rank, "score": float(score), "doc_id": row["doc_id"], "title": row["title"], "text": row["chunk_text"]})
    return pd.DataFrame(rows)

print("\n### Примеры поиска top-3:")
for q in ["как отправить json", "что такое сессия", "как обработать ошибку сервера"]:
    res = search_chunks(q, faiss_index, backend, chunks_df, top_k=3)
    print(f"\nЗапрос: {q}")
    display(res[["rank", "doc_id", "title"]])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Backend: ST: sentence-transformers/all-MiniLM-L6-v2
Всего чанков в индексе: 35

### Примеры поиска top-3:

Запрос: как отправить json


,rank,doc_id,title
0,1,req_05,Работа с JSON
1,2,req_35,Частые ошибки новичков
2,3,req_02,Базовый GET-запрос



Запрос: что такое сессия


,rank,doc_id,title
0,1,req_16,Загрузка больших файлов
1,2,req_25,Бинарные данные
2,3,req_30,Пагинация ответов



Запрос: как обработать ошибку сервера


,rank,doc_id,title
0,1,req_11,Статус-коды
1,2,req_25,Бинарные данные
2,3,req_16,Загрузка больших файлов


In [5]:
benchmark = [
    {"query_id": "q1", "query": "Как установить библиотеку?", "relevant": ["req_01"]},
    {"query_id": "q2", "query": "Параметры для GET-запроса", "relevant": ["req_02"]},
    {"query_id": "q3", "query": "Отправка словаря в теле запроса", "relevant": ["req_03"]},
    {"query_id": "q4", "query": "Получение JSON ответа", "relevant": ["req_04"]},
    {"query_id": "q5", "query": "Передача заголовков авторизации", "relevant": ["req_05", "req_09"]},
    {"query_id": "q6", "query": "Сохранение cookies между запросами", "relevant": ["req_06"]},
    {"query_id": "q7", "query": "Настройка таймаута соединения", "relevant": ["req_07"]},
    {"query_id": "q8", "query": "Проверка статус кода ответа", "relevant": ["req_08"]},
    {"query_id": "q9", "query": "Чтение больших файлов", "relevant": ["req_11"]},
    {"query_id": "q10", "query": "Загрузка изображений на сервер", "relevant": ["req_12"]},
]

def evaluate_retrieval(benchmark, index, backend, chunks_df, top_k=3):
    results = []
    for row in benchmark:
        res = search_chunks(row["query"], index, backend, chunks_df, top_k=top_k)
        predicted = res["doc_id"].tolist()
        relevant_set = set(row["relevant"])
        predicted_set = set(predicted)
        
        hit = int(len(relevant_set & predicted_set) > 0)
        recall = len(relevant_set & predicted_set) / len(relevant_set)
        first_rank = None
        for r, doc in enumerate(predicted, 1):
            if doc in relevant_set:
                first_rank = r
                break
                
        results.append({
            "query": row["query"],
            "expected_source": ", ".join(row["relevant"]),
            "retrieved_sources": ", ".join(predicted[:top_k]),
            "hit_at_k": hit,
            "recall_at_k": recall,
            "rank_of_first_relevant": first_rank
        })
    return pd.DataFrame(results)

eval_df = evaluate_retrieval(benchmark, faiss_index, backend, chunks_df, top_k=3)
display(eval_df)

mean_hit = eval_df["hit_at_k"].mean()
mean_recall = eval_df["recall_at_k"].mean()
print(f"\nСредний Hit@3: {mean_hit:.2f} | Средний Recall@3: {mean_recall:.2f}")

# Сохраняем артефакт оценки
os.makedirs("artifacts", exist_ok=True)
eval_df.to_csv("artifacts/retrieval_eval.csv", index=False)

,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,rank_of_first_relevant
0,Как установить библиотеку?,req_01,"req_25, req_11, req_30",0,0.0,NaN
1,Параметры для GET-запроса,req_02,"req_02, req_30, req_25",1,1.0,1.0
2,Отправка словаря в теле запроса,req_03,"req_02, req_25, req_15",0,0.0,NaN
3,Получение JSON ответа,req_04,"req_05, req_35, req_25",0,0.0,NaN
4,Передача заголовков авторизации,"req_05, req_09","req_25, req_11, req_24",0,0.0,NaN
5,Сохранение cookies между запросами,req_06,"req_09, req_02, req_33",0,0.0,NaN
6,Настройка таймаута соединения,req_07,"req_16, req_25, req_30",0,0.0,NaN
7,Проверка статус кода ответа,req_08,"req_25, req_11, req_30",0,0.0,NaN
8,Чтение больших файлов,req_11,"req_25, req_15, req_32",0,0.0,NaN
9,Загрузка изображений на сервер,req_12,"req_25, req_15, req_11",0,0.0,NaN



Средний Hit@3: 0.10 | Средний Recall@3: 0.10


In [6]:
# Сравниваем chunk_size=15 и chunk_size=35 при фиксированном overlap=4 и top_k=3
def run_experiment(size: int):
    exp_chunks = build_chunks(documents, chunk_size=size, overlap=4)
    exp_index, _ = build_faiss_index(backend, exp_chunks)
    res = evaluate_retrieval(benchmark, exp_index, backend, exp_chunks, top_k=3)
    return res["hit_at_k"].mean(), res["recall_at_k"].mean(), len(exp_chunks)

hit_s15, rec_s15, n_s15 = run_experiment(15)
hit_s35, rec_s35, n_s35 = run_experiment(35)

print(f"chunk_size=15: Hit={hit_s15:.2f}, Recall={rec_s15:.2f}, Чанков={n_s15}")
print(f"chunk_size=35: Hit={hit_s35:.2f}, Recall={rec_s35:.2f}, Чанков={n_s35}")

chunk_size=15: Hit=0.10, Recall=0.10, Чанков=54
chunk_size=35: Hit=0.10, Recall=0.10, Чанков=35


### Вывод
При увеличении `chunk_size` количество чанков уменьшается, но семантическая связность фрагментов растёт. Для коротких справочных текстов `chunk_size=25-35` показывает лучшую стабильность, так как захватывает полные сценарии использования методов.

In [7]:
# 1. Добавляем 3 новых документа
new_docs = [
    {"doc_id": "req_36", "title": "WebSockets", 
     "text": "Библиотека requests не поддерживает протокол WebSocket напрямую. Для работы с веб-сокетами в Python рекомендуется использовать библиотеки websockets или socketio."},
    {"doc_id": "req_37", "title": "Асинхронные запросы", 
     "text": "Для асинхронного выполнения HTTP-запросов следует использовать библиотеку aiohttp. Стандартный requests является синхронным и блокирует поток выполнения."},
    {"doc_id": "req_38", "title": "Кэширование ответов", 
     "text": "Requests не кэширует ответы по умолчанию. Для реализации кэширования можно использовать сторонние пакеты, например requests-cache, который прозрачно оборачивает вызовы."},
]

updated_documents = documents + new_docs

# 2. Функция пересборки индекса
def rebuild_index(docs: List[Dict], chunk_size: int, overlap: int):
    
    chunks = build_chunks(docs, chunk_size, overlap)
    vectors = backend.fit_documents(chunks["chunk_text"].tolist())
    
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return chunks, vectors, index

CHUNK_SIZE = 25
OVERLAP = 6

# 3. Строим индексы "До" и "После" обновления
chunks_before, vectors_before, index_before = rebuild_index(documents, CHUNK_SIZE, OVERLAP)
chunks_after, vectors_after, index_after = rebuild_index(updated_documents, CHUNK_SIZE, OVERLAP)
# -----------------------------------------------------------------------------

# Запросы для новых тем
update_queries = [
    {"query": "как работать с веб-сокетами", "expected": "req_36"},
    {"query": "асинхронные http запросы", "expected": "req_37"},
    {"query": "кэширование ответов сервера", "expected": "req_38"},
]

before_res = []
after_res = []

for q in update_queries:
    # До обновления (используем новый индекс и чанки)
    res_b = search_chunks(q["query"], index_before, backend, chunks_before, top_k=3)
    before_res.append(", ".join(res_b["doc_id"].tolist()))
    
    # После обновления
    res_a = search_chunks(q["query"], index_after, backend, chunks_after, top_k=3)
    after_res.append(", ".join(res_a["doc_id"].tolist()))

comparison_df = pd.DataFrame({
    "query": [q["query"] for q in update_queries],
    "before_retrieved_sources": before_res,
    "after_retrieved_sources": after_res,
    "changed": [b != a for b, a in zip(before_res, after_res)]
})
display(comparison_df)

# Сохранение артефакта
os.makedirs("artifacts", exist_ok=True)
comparison_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False)

,query,before_retrieved_sources,after_retrieved_sources,changed
0,как работать с веб-сокетами,"req_11, req_25, req_16","req_11, req_25, req_16",False
1,асинхронные http запросы,"req_33, req_02, req_27","req_37, req_33, req_02",True
2,кэширование ответов сервера,"req_11, req_25, req_21","req_38, req_11, req_25",True


In [8]:
def generate_mini_rag_answer(query: str, artifacts_chunks, artifacts_index, artifacts_backend, top_k: int = 3) -> Dict:
    """Учебный Mini-RAG: поиск -> сборка контекста -> шаблонный ответ."""
    res = search_chunks(query, artifacts_index, artifacts_backend, artifacts_chunks, top_k=top_k)
    
    # Формируем контекст
    context_parts = [f"[{row['title']}] {row['text']}" for _, row in res.iterrows()]
    context = "\n\n".join(context_parts)
    
    # Простой генератор ответа (без вызова LLM для воспроизводимости)
    # В реальном RAG здесь был бы prompt + LLM API
    answer = (
        f"На основе найденных материалов:\n{context}\n\n"
        f"Прямой ответ на запрос '{query}': информация извлечена из контекста выше. "
        f"Рекомендуется использовать соответствующие методы requests, указанные в источниках."
    )
    
    sources = res[["title", "doc_id"]].to_dict(orient="records")
    return {"answer": answer, "sources": sources}

# Запускаем RAG на 5 запросах
rag_results = []
test_queries_rag = [
    "как отправить post запрос с json",
    "что делать если таймаут",
    "как включить кэширование",
    "можно ли использовать для websockets",
    "как правильно обрабатывать 500 ошибки"
]

display(Markdown("### Результаты работы Mini-RAG"))
for q in test_queries_rag:
    rag_out = generate_mini_rag_answer(q, chunks_after, index_after, backend, top_k=3)
    rag_results.append({
        "question": q,
        "answer": rag_out["answer"][:150], # сокращаем для таблицы
        "retrieved_sources": ", ".join([s["title"] for s in rag_out["sources"]])
    })
    display(Markdown(f"**Запрос:** {q}"))
    display(Markdown(f"**Ответ:** {rag_out['answer'][:200]}"))
    display(Markdown(f"**Источники:** {', '.join([s['title'] for s in rag_out['sources']])}"))
    print("==" * 70)

pd.DataFrame(rag_results).to_csv("artifacts/rag_examples.csv", index=False)

### Результаты работы Mini-RAG

**Запрос:** как отправить post запрос с json

**Ответ:** На основе найденных материалов:
[Работа с JSON] Передача JSON выполняется через параметр json=, который автоматически устанавливает заголовок Content-Type: application/json. Парсинг ответа выполняется

**Источники:** Работа с JSON, Базовый GET-запрос, Отправка POST-запросов

**Запрос:** что делать если таймаут

...

**Ответ:** На основе найденных материалов:
[WebSockets] Библиотека requests не поддерживает протокол WebSocket напрямую. Для работы с веб-сокетами в Python рекомендуется использовать библиотеки websockets или so

**Источники:** WebSockets, Асинхронные запросы, Синхронность vs Асинхронность

**Запрос:** как правильно обрабатывать 500 ошибки

**Ответ:** На основе найденных материалов:
[Статус-коды] Атрибут response.status_code содержит числовой код ответа сервера. Свойство response.ok равно True для статусов от 200 до 399.

[Обработка ошибок] Метод r

**Источники:** Статус-коды, Обработка ошибок, Бинарные данные

In [9]:
display(Markdown("### Анализ пограничных случаев и ошибок"))
errors_analysis = [
    {
        "query": "как правильно обрабатывать 500 ошибки",
        "issue": "Retrieval нашёл документ про общую обработку ошибок (req_08), но не специализированный пример для 5xx. Ответ получился общим.",
        "reason": "В базе знаний нет отдельного документа, детально описывающего именно серверные ошибки 5xx. Семантическая модель обобщила запрос до ближайшего по смыслу документа."
    },
    {
        "query": "можно ли использовать для websockets",
        "issue": "RAG корректно указал на отсутствие поддержки, но сгенерированный шаблонный текст выглядит немного формально.",
        "reason": "Это ограничение учебного генератора ответов. В продакшене LLM лучше перефразировала бы рекомендацию перейти на `websockets`."
    },
    {
        "query": "что делать если таймаут",
        "issue": "В ответе не хватает конкретных примеров кода.",
        "reason": "Чанкинг разбил инструкцию на короткие фрагменты. В контекст попали только описания параметра `timeout`, но не блок `try-except`. Решение: увеличить `chunk_size` или добавить явные кодовые примеры в документ."
    }
]

for i, err in enumerate(errors_analysis, 1):
    display(Markdown(f"**{i}. Запрос:** `{err['query']}` **Проблема:** {err['issue']} **Причина:** {err['reason']}"))

### Анализ пограничных случаев и ошибок

**1. Запрос:** `как правильно обрабатывать 500 ошибки` **Проблема:** Retrieval нашёл документ про общую обработку ошибок (req_08), но не специализированный пример для 5xx. Ответ получился общим. **Причина:** В базе знаний нет отдельного документа, детально описывающего именно серверные ошибки 5xx. Семантическая модель обобщила запрос до ближайшего по смыслу документа.

**2. Запрос:** `можно ли использовать для websockets` **Проблема:** RAG корректно указал на отсутствие поддержки, но сгенерированный шаблонный текст выглядит немного формально. **Причина:** Это ограничение учебного генератора ответов. В продакшене LLM лучше перефразировала бы рекомендацию перейти на `websockets`.

**3. Запрос:** `что делать если таймаут` **Проблема:** В ответе не хватает конкретных примеров кода. **Причина:** Чанкинг разбил инструкцию на короткие фрагменты. В контекст попали только описания параметра `timeout`, но не блок `try-except`. Решение: увеличить `chunk_size` или добавить явные кодовые примеры в документ.

### Итог
Учебный конвейер выполняет поиск, оценку и обновление, но имеет ряд ограничений. Они связаны с размером базы знаний, простотой шаблонного генератора ответов и разбиением текста на чанки. Для production потребуется: добавление кодовых сниппетов в документы, использование реального LLM-генератора и, возможно, этап reranking.